# Job Salary Prediction Using Statistical Inference and Machine Learning

**Student:** Shreya Malewar
**Dataset:** `job_salary_prediction_dataset.csv`  
**Target:** `salary` (annual salary)

## Project aim
Analyze which job, education, experience, skill, company, location, and work-mode factors are associated with salary. Apply statistical-inference practicals, compare regression models, and build a small salary-prediction interface.

### Important notes
- This notebook is designed for Google Colab, but also works in a local Jupyter environment.
- The supplied dataset contains 250,000 rows and 10 columns with no missing values.
- Salary values are treated as annual monetary amounts in the dataset's units.
- The median salary is used as a project-defined threshold for the chi-square analysis.
- The paired t-test compares actual and predicted salaries for the same held-out jobs; it is not a replacement for MAE, RMSE, and R².


## 1. Import libraries and load the dataset

`pandas` and `numpy` manage data, `matplotlib` and `seaborn` create graphs, `scipy.stats` provides statistical tests, and `scikit-learn` provides preprocessing and regression models. The upload cell works in Colab; locally, place the CSV beside the notebook or update `DATA_PATH`.

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
sns.set_theme(style="whitegrid", palette="Set2")
RANDOM_STATE = 42
df = pd.read_csv("job_salary_prediction_dataset.csv")
print("Dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nFirst five rows:")
display(df.head())


## 2. Check data quality and clean the dataset

This section checks types, missing values, duplicates, and invalid numeric values. Exact duplicates are removed. The imputation code is defensive: it will still work if a future version of the file contains missing values.

In [ ]:
print("Data types:"); display(df.dtypes.to_frame("dtype"))
print("Missing values before cleaning:"); display(df.isna().sum().to_frame("Missing count"))
print("Duplicate rows before cleaning:", df.duplicated().sum())
numeric_cols = ["experience_years", "skills_count", "certifications", "salary"]
categorical_cols = ["job_title", "education_level", "industry", "company_size", "location", "remote_work"]
df = df.drop_duplicates().copy()
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(df[col].median())
for col in categorical_cols:
    df[col] = df[col].astype("string").fillna(df[col].mode(dropna=True).iloc[0])
df = df[(df.experience_years >= 0) & (df.skills_count >= 0) & (df.certifications >= 0) & (df.salary > 0)].copy()
print("\nShape after cleaning:", df.shape)
print("Remaining missing values:", int(df.isna().sum().sum()))
print("Remaining duplicate rows:", int(df.duplicated().sum()))


## 3. Descriptive statistics

Mean, median, mode, variance, standard deviation, and quartiles summarize numerical variables. The median is especially useful when salary values are not symmetric.

In [ ]:
display(df[numeric_cols].describe().T)
for col in numeric_cols:
    print(f"\n{col}: mean={df[col].mean():,.2f}, median={df[col].median():,.2f}, mode={df[col].mode().iloc[0]}, variance={df[col].var():,.2f}, std={df[col].std():,.2f}")
print("\nCategorical summaries:")
for col in categorical_cols:
    print(f"\n{col}"); display(df[col].value_counts().head(15).rename_axis(col).to_frame("count"))
salary_sample = df.salary.sample(n=min(1000, len(df)), random_state=RANDOM_STATE)
print("\nRandom sample size:", len(salary_sample), "Sample mean:", round(salary_sample.mean(),2), "Sample median:", round(salary_sample.median(),2))


## 4. Exploratory Data Analysis (EDA) and graphs

Histograms show distributions, boxplots show spread and outliers, scatterplots show numerical relationships, and grouped boxplots compare salary across categories. A 20,000-row plotting sample is used only for readability.

In [ ]:
plot_sample = df.sample(n=min(20000, len(df)), random_state=RANDOM_STATE)
fig, axes = plt.subplots(2,2,figsize=(14,9))
for ax,col in zip(axes.ravel(),numeric_cols): sns.histplot(plot_sample[col],bins=30,kde=True,ax=ax); ax.set_title(f"Distribution of {col}")
plt.tight_layout(); plt.show()
plt.figure(figsize=(10,4)); sns.boxplot(x=plot_sample.salary); plt.title("Salary boxplot"); plt.show()
fig,axes=plt.subplots(1,2,figsize=(15,5))
sns.scatterplot(data=plot_sample,x="experience_years",y="salary",alpha=.25,s=18,ax=axes[0]); axes[0].set_title("Salary vs experience")
sns.scatterplot(data=plot_sample,x="skills_count",y="salary",alpha=.25,s=18,ax=axes[1]); axes[1].set_title("Salary vs skills count")
plt.tight_layout(); plt.show()
fig,axes=plt.subplots(1,3,figsize=(18,5))
for ax,col in zip(axes,["education_level","company_size","remote_work"]): sns.boxplot(data=plot_sample,x=col,y="salary",ax=ax); ax.tick_params(axis="x",rotation=30)
plt.tight_layout(); plt.show()


## 5. Probability distributions, sampling, and the Central Limit Theorem

The observed salary distribution is compared with example uniform and binomial distributions. Repeated salary samples illustrate that sample means tend toward an approximately normal distribution as sample size increases.

In [ ]:
plt.figure(figsize=(8,4)); sns.histplot(df.salary,bins=40,kde=True); plt.title("Observed salary distribution"); plt.show()
rng=np.random.default_rng(RANDOM_STATE)
plt.figure(figsize=(7,4)); plt.hist(rng.uniform(0,1,1000),bins=20,edgecolor="black"); plt.title("Simulated uniform distribution"); plt.show()
plt.figure(figsize=(7,4)); plt.hist(rng.binomial(10,.5,1000),bins=np.arange(-.5,11.5,1),edgecolor="black"); plt.title("Simulated binomial distribution"); plt.show()


In [ ]:
sample=df.salary.sample(n=min(1000,len(df)),random_state=RANDOM_STATE)
print("Population mean salary:",round(df.salary.mean(),2)); print("Random sample mean salary:",round(sample.mean(),2))
sample_size=min(100,len(df)); sample_means=[rng.choice(df.salary.to_numpy(),size=sample_size,replace=True).mean() for _ in range(1000)]
plt.figure(figsize=(8,4)); sns.histplot(sample_means,bins=30,kde=True); plt.axvline(df.salary.mean(),color="red",linestyle="--",label="Population mean"); plt.title(f"Sampling distribution of salary means (n={sample_size})"); plt.legend(); plt.show()
print("Mean of simulated sample means:",round(np.mean(sample_means),2))


## 6. Sampling techniques and standard error

Simple random sampling gives each record a chance of selection. Stratified sampling takes observations within education groups. Standard error estimates the variation of a sample mean.

In [ ]:
simple_sample=df.sample(n=min(1000,len(df)),random_state=RANDOM_STATE)
stratified_sample=(df.groupby("education_level",group_keys=False).apply(lambda g:g.sample(n=min(200,len(g)),random_state=RANDOM_STATE)).reset_index(drop=True))
print("Simple random sample size:",len(simple_sample)); print("Stratified sample size:",len(stratified_sample)); display(stratified_sample.groupby("education_level").salary.agg(["count","mean","std"]).round(2))
print("Standard error of full-data mean:",round(df.salary.std(ddof=1)/np.sqrt(len(df)),2))


## 7. Confidence interval for the mean salary

A 95% t-based confidence interval estimates a plausible range for the population mean represented by this dataset. It does not contain 95% of individual salaries.

In [ ]:
confidence=.95; mean_salary=df.salary.mean(); standard_error=stats.sem(df.salary)
ci_low,ci_high=stats.t.interval(confidence,df=len(df)-1,loc=mean_salary,scale=standard_error)
print(f"Mean salary: {mean_salary:,.2f}"); print(f"95% confidence interval: ({ci_low:,.2f}, {ci_high:,.2f})")


## 8. Hypothesis tests (significance level α = 0.05)

A p-value below 0.05 leads to rejection of the null hypothesis. Interpret significance together with effect size, sample size, and assumptions.

In [ ]:
alpha=.05; reference_salary=150000
t_stat,p_value=stats.ttest_1samp(df.salary,popmean=reference_salary)
print("ONE-SAMPLE T-TEST: H0: mean salary = 150,000"); print(f"t={t_stat:.4f}, p={p_value:.6g};", "Reject H0" if p_value<alpha else "Fail to reject H0")


In [ ]:
usa=df.loc[df.location=="USA","salary"]; india=df.loc[df.location=="India","salary"]
t_stat,p_value=stats.ttest_ind(usa,india,equal_var=False)
print("WELCH TWO-SAMPLE T-TEST: USA vs India"); print(f"USA mean={usa.mean():,.2f}; India mean={india.mean():,.2f}"); print(f"t={t_stat:.4f}, p={p_value:.6g};", "Reject H0" if p_value<alpha else "Fail to reject H0")


In [ ]:
education_groups=[g.salary.to_numpy() for _,g in df.groupby("education_level") if len(g)>1]
f_stat,p_value=stats.f_oneway(*education_groups)
print("ONE-WAY ANOVA: equal mean salaries across education levels"); print(f"F={f_stat:.4f}, p={p_value:.6g};", "Reject H0" if p_value<alpha else "Fail to reject H0")


In [ ]:
median_salary=df.salary.median(); df["Salary_Category"]=np.where(df.salary>=median_salary,"At/above median","Below median")
contingency=pd.crosstab(df.education_level,df.Salary_Category); chi2,p_value,dof,expected=stats.chi2_contingency(contingency)
print("CHI-SQUARE TEST: education level vs salary category"); display(contingency); print(f"Threshold={median_salary:,.2f}; chi2={chi2:.4f}, dof={dof}, p={p_value:.6g};", "Reject H0" if p_value<alpha else "Fail to reject H0")


## 9. Correlation and significance tests

Pearson correlation measures linear association; Spearman correlation measures monotonic association using ranks. Correlation does not establish causation, and very large samples can make small effects statistically significant.

In [ ]:
correlation_cols=["experience_years","skills_count","certifications","salary"]
display(df[correlation_cols].corr().round(4))
for x in ["experience_years","skills_count","certifications"]:
    pr,pp=stats.pearsonr(df[x],df.salary); sr,sp=stats.spearmanr(df[x],df.salary)
    print(f"{x} vs salary: Pearson r={pr:.4f}, p={pp:.6g}; Spearman rho={sr:.4f}, p={sp:.6g}")


## 10. Prepare data for machine learning

Categorical predictors are one-hot encoded and numeric predictors are standardized inside a single pipeline. To keep Colab responsive, a reproducible 50,000-row sample is used for model training; the statistical analysis above uses all cleaned records.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
import pandas as pd

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=10.0),
    "Lasso Regression": Lasso(alpha=100.0, max_iter=5000),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}
results = []
trained_models = {}

for name, estimator in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "R2": r2_score(y_test, pred)
    })

    trained_models[name] = pipe

# Highest R2 (closest to 1) first
results_df = (
    pd.DataFrame(results)
    .sort_values("R2", ascending=False)
    .reset_index(drop=True)
)

display(
    results_df.style.format({
        "MAE": "{:,.2f}",
        "RMSE": "{:,.2f}",
        "R2": "{:.4f}"
    })
)

# Best model = the one with the highest R2 (row 0 after the descending sort)
best_name = results_df.loc[0, "Model"]
best_model = trained_models[best_name]

print("Best model by R2:", best_name)
print(f"Best R2: {results_df.loc[0, 'R2']:.4f}")


## 11. Train and compare regression models

Linear Regression is the baseline. Ridge adds L2 regularization and Lasso adds L1 regularization. MAE is average absolute error, RMSE penalizes larger errors, and R² is explained variance.

In [ ]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
models={"Linear Regression":LinearRegression(),"Ridge Regression":Ridge(alpha=10.0),"Lasso Regression":Lasso(alpha=100.0,max_iter=5000)}
results=[]; trained_models={}
for name,estimator in models.items():
    pipe=Pipeline([("preprocessor",preprocessor),("model",estimator)]); pipe.fit(X_train,y_train); pred=pipe.predict(X_test)
    results.append({"Model":name,"MAE":mean_absolute_error(y_test,pred),"RMSE":mean_squared_error(y_test,pred)**.5,"R2":r2_score(y_test,pred)}); trained_models[name]=pipe
results_df=pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True); display(results_df.style.format({"MAE":"{:,.2f}","RMSE":"{:,.2f}","R2":"{:.4f}"}))
best_name=results_df.loc[0,"Model"]; best_model=trained_models[best_name]; print("Best model by RMSE:",best_name)


In [ ]:
best_predictions=best_model.predict(X_test)
plt.figure(figsize=(7,6)); sns.scatterplot(x=y_test,y=best_predictions,alpha=.25,s=18); lo=min(y_test.min(),best_predictions.min()); hi=max(y_test.max(),best_predictions.max()); plt.plot([lo,hi],[lo,hi],"r--",label="Perfect prediction"); plt.xlabel("Actual salary"); plt.ylabel("Predicted salary"); plt.title(f"Actual vs predicted salary: {best_name}"); plt.legend(); plt.show()


## 12. Paired t-test: actual vs. predicted salaries

Each prediction is paired with the actual salary for the same test record, so `stats.ttest_rel()` is appropriate. The test asks whether the average prediction error differs from zero.

In [ ]:
t_stat,p_value=stats.ttest_rel(y_test.to_numpy(),best_predictions)
print("PAIRED T-TEST: Actual vs Predicted Salary"); print(f"Mean actual={y_test.mean():,.2f}; Mean predicted={best_predictions.mean():,.2f}"); print(f"t={t_stat:.4f}, p={p_value:.6g};", "Reject H0" if p_value<alpha else "Fail to reject H0")


## 13. Save the best model

The complete pipeline is saved so future inputs receive the same encoding, scaling, and regression treatment.

In [ ]:
import joblib
import pickle

model_filename = "best_job_salary_model.joblib"
pkl_filename = "best_job_salary_model.pkl"

# Save as .joblib
joblib.dump(best_model, model_filename)
print("Saved model:", model_filename)

# Save as .pkl
with open(pkl_filename, "wb") as f:
    pickle.dump(best_model, f)
print("Saved model:", pkl_filename)

# In Colab, uncomment to download both files:
# from google.colab import files
# files.download(model_filename)
# files.download(pkl_filename)

## 14. Interactive salary prediction interface using Gradio

The interface collects the same fields used by the model and returns a predicted salary. Install Gradio only if it is not available in the notebook environment.

In [ ]:
# Optional Colab install: !pip -q install gradio


In [ ]:
try:
    import gradio as gr
    loaded_model=joblib.load("best_job_salary_model.joblib")
    def predict_salary(job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications):
        row=pd.DataFrame([locals()]); return f"Estimated salary: {loaded_model.predict(row)[0]:,.0f}"
    demo=gr.Interface(fn=predict_salary,inputs=[gr.Dropdown(sorted(df.job_title.unique()),label="Job title"),gr.Slider(0,int(df.experience_years.max()),value=5,step=1,label="Experience years"),gr.Dropdown(sorted(df.education_level.unique()),value="Bachelor",label="Education"),gr.Slider(0,int(df.skills_count.max()),value=10,step=1,label="Skills count"),gr.Dropdown(sorted(df.industry.unique()),label="Industry"),gr.Dropdown(sorted(df.company_size.unique()),label="Company size"),gr.Dropdown(sorted(df.location.unique()),label="Location"),gr.Dropdown(sorted(df.remote_work.unique()),label="Remote work"),gr.Slider(0,int(df.certifications.max()),value=2,step=1,label="Certifications")],outputs=gr.Textbox(label="Prediction"),title="Job Salary Predictor")
    print("Gradio interface created. Run demo.launch() to open it.")
    # demo.launch()
except ImportError:
    print("Gradio is not installed. Uncomment the install cell above to enable the interface.")


## 15. Final conclusion template

After running the notebook, report the mean, median, standard deviation, and 95% confidence interval for salary. Explain the distribution. Report the one-sample t-test, USA-versus-India Welch test, education ANOVA, chi-square test, and correlation values with interpretations. Compare Linear, Ridge, and Lasso using MAE, RMSE, and R², identify the best held-out-test model, and discuss limitations: observational data cannot prove causality; important salary variables may be absent; location may encode market or currency differences; and huge samples make small effects statistically significant.